<a href="https://colab.research.google.com/github/zlerdworatawee/iclue-scripts/blob/main/Orth_Copy_of_RSK_Linear_OP_mnd_var.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.display import display, Math
import numpy as np
import pandas as pd
import sympy as sp
from sympy import symbols, sympify, Matrix, Eq, solve, expand, Abs, pprint, factor, Mul
from itertools import permutations
import scipy
import math
from functools import lru_cache

# Attempt at Coding Algorithm A

Change pi and sig below to test RSK_(pi,sig) col/sum pairs


---



## Functions Reference:
### Functions for RSK_{pi,sig}
```
  - contingency_tables(row_sums, col_sums):
      - @param row_sums:
      - @param col_sums:
      - Calculates the contingency tables for given row and column sum vectors based on the n-Queens problem (backtracking)

  - generate_monomials(tables, sig, pi):
      - @param tables
      - @param sig
      - @param pi

  - rsk_insert(tableau, x):
  - viennot_rsk(biword):
  - print_tableau(tableau, name='T'):
  - determinant():
  - generate_final_matrix():
  - foo():
  - display_matrix():
```


---


### Functions for RSK_{m,n,d}
```
  - contingency_vectors():
  - generate_matrices():
  -
```



In [ ]:
def contingency_tables(row_sums, col_sums):
    '''
    This function generates the contingency tables
    for a given set of row and column sums
    using a backtracking/recursive approach.
    '''
    m, n = len(row_sums), len(col_sums)
    result = []

    def backtrack(row, table, col_sums_left):
        if row == m:
            if all(c == 0 for c in col_sums_left):
                result.append(np.array(table))
            return

        def valid_next_rows(rsum, cols_left, partial=[]):
            if len(partial) == n:
                if sum(partial) == rsum:
                    yield list(partial)
                return
            i = len(partial)
            max_entry = min(rsum - sum(partial), cols_left[i])
            for x in range(max_entry + 1):
                yield from valid_next_rows(rsum, cols_left, partial + [x])

        for row_vals in valid_next_rows(row_sums[row], col_sums_left):
            new_cols = [c - x for c, x in zip(col_sums_left, row_vals)]
            if all(c >= 0 for c in new_cols):
                backtrack(row + 1, table + [row_vals], new_cols)

    backtrack(0, [], col_sums)
    return result

In [ ]:
def generate_monomials(tables, pi, sig):
    '''
    This function simply fills out the above dictionaries and lists
    @params tables is a list of contingency tables generated by the above function
    '''
    m, n = len(sig), len(pi)

    # Create symbols z_ij for monomial basis
    z = Matrix(m, n, lambda i, j: symbols(f"z{i+1}{j+1}"))

    monomials = []  # monomials[i] returns the monomial generated for the ith contingency table
    idx_dict = {} # idx_dict[i] returns the sequence of bump chains to be inserted for the ith contingency table
    exp_dict = {} # exp_dict[i] returns a a dictionary of monomials paired with their respective exponent for the ith contingency table

    idx = 0 # dummy counter
    for table in tables:
        mon = 1
        temp_idx = []
        temp_exp_idx = []
        table_dict = {}
        for j in range(m): # j is row index
            for i in range(n): # i is column index
                exp = table[i][j] # Original, incorrect access
                if exp > 0:
                    pair = (i + 1, j + 1) # Original, incorrect pair
                    temp_idx.extend([pair] * exp)
                    mon *= z[i, j]**exp # Original, incorrect access
                    if z[i, j] not in table_dict:
                        table_dict[z[i, j]] = exp
                    else:
                        table_dict[z[i, j]] += exp
        exp_dict[idx] = table_dict
        idx_dict[idx] = temp_idx

        idx += 1
        monomials.append(mon)

    return monomials, idx_dict, exp_dict, z

In [ ]:
def rsk_insert(tableau, x):
    tableau = tableau.copy()
    rows, cols = tableau.shape
    bumped = x
    for r in range(rows):
        row = tableau[r]
        mask = (row > 0)
        eligible = row[mask]
        idx = np.where(eligible > bumped)[0]
        if idx.size == 0:
            insert_pos = np.sum(mask)
            if insert_pos < cols:
                tableau[r, insert_pos] = bumped
                return tableau, (r, insert_pos)
            else:
                continue
        else:
            i = idx[0]
            bumped, tableau[r, i] = tableau[r, i], bumped
    empty_row = np.zeros(cols, dtype=int)
    empty_row[0] = bumped
    tableau = np.vstack([tableau, empty_row])
    return tableau, (tableau.shape[0] - 1, 0)

def viennot_rsk(biword):
    n = len(biword)
    P = np.zeros((n, n), dtype=int)
    Q = np.zeros((n, n), dtype=int)
    for a, b in biword:
        P, (r, c) = rsk_insert(P, a)
        Q[r, c] = b
    return P, Q

def print_tableau(tableau, name='T'):
    '''
    for debugging
    @params tableau is a numpy array
    @params name is a string
    '''
    print(f"{name}:")
    for row in tableau:
        row_nonzero = row[row > 0]
        if row_nonzero.size > 0:
            print(row_nonzero)

In [ ]:
def determinant(P, Q, z_matrix):
  '''
  This function generates the determinant
  for the given RSK correspondence tableaus
  P and Q. @params idx refers to the ith
  contingency table.
  '''
  det_collection = []

  for col_idx in range(P.shape[1]):
      n_val = np.count_nonzero(P[:, col_idx]) # Renamed local n to n_val to avoid confusion
      p_col = P[:, col_idx]
      q_col = Q[:, col_idx]
      det = Matrix.zeros(n_val,n_val)
      for i, p_val in enumerate(p_col):
        for j, q_val in enumerate(q_col):
          if p_val != 0 and q_val != 0:
            key = z_matrix[p_val - 1, q_val - 1]
            det[i, j] = key
      det_collection.append(det)

  expr = 1
  for det in det_collection:
    minor_det = det.det()
    expr *= minor_det
  expr = expand(expr)
  return expr

In [ ]:
def generate_final_matrix(monomial_mapping, determinant_list):
    '''
    This block of code uses the initial
    dictionary to fill out the final matrix
    '''

    final_matrix = np.zeros((len(monomial_mapping), len(monomial_mapping)))
    final_matrix

    final_size = len(monomial_mapping)
    for col_idx in range(final_size):
        curr_list = determinant_list[col_idx]
        final_slice = final_matrix[:, col_idx]

        for term in curr_list.as_ordered_terms():
          coeff, monomial = term.as_coeff_Mul()

          if monomial in monomial_mapping:
              row_idx = monomial_mapping[monomial]
              final_slice[row_idx] = coeff

          elif -monomial in monomial_mapping:
              row_idx = monomial_mapping[-monomial]
              final_slice[row_idx] = -coeff

        final_matrix[:, col_idx] = final_slice
    return final_matrix

In [ ]:
def foo(matrix):
        '''
        function that may be a bit faster than np.linalg.eig()
        @params matrix is a numpy array
        '''
        eigenvalues, eigenvectors = np.linalg.eig(matrix)
        for eigenvalue in np.unique(eigenvalues):
            algebraic_multiplicity = np.sum(eigenvalues == eigenvalue)

            eigenvectors_for_eigenvalue = eigenvectors[:, eigenvalues == eigenvalue]
            geometric_multiplicity = np.linalg.matrix_rank(eigenvectors_for_eigenvalue)

            if algebraic_multiplicity != geometric_multiplicity:
                return False
        return True

def display(final_matrix):
    print(f"Sigma: {sig}")
    print(f"Pi: {pi}")
    print(f"Trace: {np.linalg.trace(final_matrix)}")
    print(f"is Diagonalizable: {foo(final_matrix)}" )
    print(f"Determinant: {np.linalg.det(final_matrix)}")
    char_poly_coeffs = np.poly(final_matrix)
    x = symbols('x')
    num_char_poly_coeffs = len(char_poly_coeffs)
    char_poly = [char_poly_coeffs[n] * x**(num_char_poly_coeffs - n - 1) for n in range(num_char_poly_coeffs)]
    pretty_eq = sp.latex(factor(sum(char_poly)))
    pprint(f"Characteristic Polynomial:")
    print(display(Math(pretty_eq)))

<!-- end of fxn code--!>

In [ ]:
3# row/col margins
# sig[i] = sum of row i
# pi[i] = sum of col i

print("If pi and sig are not the same length, please pad the smaller array with zeroes")

# pi = [int(x) for x in input("Enter pi: ").split(",")]
# sig = [int(x) for x in input("Enter sig: ").split(",")]
pi = [2, 1, 1]
sig = [1, 2, 1]
# pi = [3, 5, 0]
# sig = [3, 2, 3]
m, n = len(sig), len(pi)

# Create symbols z_ij for monomial basis
z = Matrix(m, n, lambda i, j: symbols(f"z{i+1}{j+1}"))

If pi and sig are not the same length, please pad the smaller array with zeroes


In [ ]:
# table sanity check
tables = contingency_tables(sig, pi)
tables = tables[::-1]
for t in tables:
    print(t, end="\n\n")

[[1 0 0]
 [1 1 0]
 [0 0 1]]

[[1 0 0]
 [1 0 1]
 [0 1 0]]

[[1 0 0]
 [0 1 1]
 [1 0 0]]

[[0 1 0]
 [2 0 0]
 [0 0 1]]

[[0 1 0]
 [1 0 1]
 [1 0 0]]

[[0 0 1]
 [2 0 0]
 [0 1 0]]

[[0 0 1]
 [1 1 0]
 [1 0 0]]



In [ ]:
# gives the correct hashing order for the ordered monomial basis
monomials, idx_dict, exp_dict, z = generate_monomials(tables, pi, sig)
print(idx_dict)
monomial_mapping = {mon: i for i, mon in enumerate(monomials)}; monomial_mapping

{0: [(1, 1), (2, 1), (2, 2), (3, 3)], 1: [(1, 1), (2, 1), (3, 2), (2, 3)], 2: [(1, 1), (3, 1), (2, 2), (2, 3)], 3: [(2, 1), (2, 1), (1, 2), (3, 3)], 4: [(2, 1), (3, 1), (1, 2), (2, 3)], 5: [(2, 1), (2, 1), (3, 2), (1, 3)], 6: [(2, 1), (3, 1), (2, 2), (1, 3)]}


{z11*z21*z22*z33: 0,
 z11*z21*z23*z32: 1,
 z11*z22*z23*z31: 2,
 z12*z21**2*z33: 3,
 z12*z21*z23*z31: 4,
 z13*z21**2*z32: 5,
 z13*z21*z22*z31: 6}

In [ ]:
numpy_tableaus = [viennot_rsk(index) for index in idx_dict.values()] # converting our bump chains(?) into Tableaus P and Q

In [ ]:
determinant_list = [determinant(P, Q, z) for i, (P, Q) in enumerate(numpy_tableaus)] # determinants generated from P,Q

In [ ]:
final_matrix = generate_final_matrix(monomial_mapping, determinant_list)

In [ ]:
from IPython.display import display, Math

final_matrix = generate_final_matrix(monomial_mapping, determinant_list)

print(f"Sigma: {sig}")
print(f"Pi: {pi}")
print(final_matrix)
print(f"Trace: {np.linalg.trace(final_matrix)}")
print(f"is Diagonalizable: {foo(final_matrix)}" )
print(f"Determinant: {np.linalg.det(final_matrix)}")
char_poly_coeffs = np.poly(final_matrix)
x = symbols('x')
num_char_poly_coeffs = len(char_poly_coeffs)
char_poly = [char_poly_coeffs[n] * x**(num_char_poly_coeffs - n - 1) for n in range(num_char_poly_coeffs)]
pretty_eq = sp.latex(factor(sum(char_poly)))
pprint(f"Characteristic Polynomial:")
print(display(Math(pretty_eq)))

Sigma: [1, 2, 1]
Pi: [2, 1, 1]
[[ 1.  1.  0.  1.  1.  0.  1.]
 [ 0.  0.  1.  0.  0.  1. -1.]
 [ 0.  0.  0.  0. -1.  0.  0.]
 [ 0.  0.  0. -1. -1.  0. -1.]
 [ 0.  0. -1.  0.  1.  0.  1.]
 [ 0.  0.  0.  0.  0. -1.  1.]
 [ 0. -1.  0.  0.  0.  0. -1.]]
Trace: -1.0
is Diagonalizable: True
Determinant: -1.0
Characteristic Polynomial:


<IPython.core.display.Math object>

None


In [ ]:
def RSK_pi_sig(r_pair: list, show=False) -> tuple:
    '''
    Compute the RSK matrix for a given (sig, pi) pair
    '''
    sig, pi = r_pair
    m, n = len(sig), len(pi)
    tables = contingency_tables(sig, pi)
    tables = tables[::-1]  # FIX: Reverse the tables to get correct ordering

    monomials, idx_dict, exp_dict, z_local = generate_monomials(tables, pi, sig)

    numpy_tableaus = [viennot_rsk(index) for index in idx_dict.values()]
    determinant_list = [determinant(P, Q, z_local) for P, Q in numpy_tableaus]
    monomial_mapping = {mon: i for i, mon in enumerate(monomials)}

    final_matrix = generate_final_matrix(monomial_mapping, determinant_list)

    if show:
        display(final_matrix)

    return final_matrix


In [ ]:
RSK_pi_sig(([2, 1, 1], [1, 1, 2]))

array([[ 1.,  1.,  1.,  0.,  0.,  1.,  0.],
       [ 0.,  0.,  0.,  1.,  0.,  0.,  1.],
       [ 0.,  0.,  0.,  0.,  1., -1., -1.],
       [ 0., -1.,  0., -1.,  0., -1., -1.],
       [ 0.,  0., -1.,  0., -1.,  0.,  1.],
       [ 0.,  0.,  0.,  0.,  0.,  1.,  1.],
       [ 0.,  0.,  0.,  0.,  0.,  0., -1.]])

# Brute Force RSK_{m,n,d}:
- @param m,n: len of pi/sig weight vectors
- @param d: degree

In [ ]:
def contingency_vectors(n, d):
    '''
    This function generates the possible contingency vectors
    @params n: len of vector
    @params d: degree
    '''
    vectors = []
    def recusion(vector, degrees_left):
        if len(vector) == n:
            vectors.append(vector)
            return
        for i in range(degrees_left + 1)[::-1]:
            new_vector = vector + [i]

            if len(new_vector) == n:  # pruning
                vectors.append(new_vector)
                return

            recusion(new_vector, degrees_left - int(i))

    recusion([], d)
    return vectors

In [ ]:
def generate_matrices(m, n, d):
    '''
    This function generates the RSK_{m,n,d} matrices
    @params m,n: len of pi/sig weight vectors can assume m = n
    @params d: degree
    '''
    contingency_vectors_list = contingency_vectors(m, d)
    contingency_vector_map = {i : tuple(vector) for i, vector in enumerate(contingency_vectors_list)}
    matrices = []
    final_dim = 0
    for pi in contingency_vector_map.values():
        for sig in contingency_vector_map.values():
            final_matrix = RSK_pi_sig((sig, pi), show=False)
            final_dim += final_matrix.shape[0]
            print(f"{pi}, {sig}")
            matrices.append(final_matrix)
    return matrices

In [ ]:
from scipy.linalg import block_diag
from IPython.display import display, Math
'''
Edit m and d below to obtain the desired RSK_{m,m,d} matrix
'''
m = 3
d = 4
matrices = generate_matrices(m,m,d)
final_rsk_matrix = block_diag(*matrices)

print(f"m: {m}")
print(f"n: {n}")
print(f"d: {d}")
print(f"Trace: {np.linalg.trace(final_rsk_matrix)}")
print(f"Determinant: {np.linalg.det(final_rsk_matrix)}")

(4, 0, 0), (4, 0, 0)
(4, 0, 0), (3, 1, 0)
(4, 0, 0), (3, 0, 1)
(4, 0, 0), (2, 2, 0)
(4, 0, 0), (2, 1, 1)
(4, 0, 0), (2, 0, 2)
(4, 0, 0), (1, 3, 0)
(4, 0, 0), (1, 2, 1)
(4, 0, 0), (1, 1, 2)
(4, 0, 0), (1, 0, 3)
(4, 0, 0), (0, 4, 0)
(4, 0, 0), (0, 3, 1)
(4, 0, 0), (0, 2, 2)
(4, 0, 0), (0, 1, 3)
(4, 0, 0), (0, 0, 4)
(3, 1, 0), (4, 0, 0)
(3, 1, 0), (3, 1, 0)
(3, 1, 0), (3, 0, 1)
(3, 1, 0), (2, 2, 0)
(3, 1, 0), (2, 1, 1)
(3, 1, 0), (2, 0, 2)
(3, 1, 0), (1, 3, 0)
(3, 1, 0), (1, 2, 1)
(3, 1, 0), (1, 1, 2)
(3, 1, 0), (1, 0, 3)
(3, 1, 0), (0, 4, 0)
(3, 1, 0), (0, 3, 1)
(3, 1, 0), (0, 2, 2)
(3, 1, 0), (0, 1, 3)
(3, 1, 0), (0, 0, 4)
(3, 0, 1), (4, 0, 0)
(3, 0, 1), (3, 1, 0)
(3, 0, 1), (3, 0, 1)
(3, 0, 1), (2, 2, 0)
(3, 0, 1), (2, 1, 1)
(3, 0, 1), (2, 0, 2)
(3, 0, 1), (1, 3, 0)
(3, 0, 1), (1, 2, 1)
(3, 0, 1), (1, 1, 2)
(3, 0, 1), (1, 0, 3)
(3, 0, 1), (0, 4, 0)
(3, 0, 1), (0, 3, 1)
(3, 0, 1), (0, 2, 2)
(3, 0, 1), (0, 1, 3)
(3, 0, 1), (0, 0, 4)
(2, 2, 0), (4, 0, 0)
(2, 2, 0), (3, 1, 0)
(2, 2, 0), (3

# Theorem 3.22 Code for R_{m, n, d} Block Decomp


In [ ]:
def trace_det_tables(max_m=4, max_d=4):
  ''' Testing fxn for generating det/trace table '''
  trace_table = [[None]*(max_d+1) for _ in range(max_m+1)]
  det_table   = [[None]*(max_d+1) for _ in range(max_m+1)]

  for m in range(1, max_m+1):
      for d in range(1, max_d+1):
          tr, det = block_decomp(m, m, d)
          trace_table[m][d] = int(tr)
          det_table[m][d]   = int(det)

  return trace_table, det_table

In [ ]:
import math
import numpy as np
from functools import lru_cache
from collections import defaultdict

# ----------------------------
# Partitions / reduced pairs
# ----------------------------

@lru_cache(None)
def partitions_by_largest_vec(n: int):
    partition_dict = defaultdict(list)

    def helper(remaining, current):
        if remaining == 0:
            partition_dict[max(current)].append(tuple(current))
            return
        for i in range(1, remaining + 1):
            helper(remaining - i, current + [i])

    helper(n, [])
    max_index = max(partition_dict, default=0)

    out = [[] for _ in range(max_index + 1)]
    for k, v in partition_dict.items():
        out[k] = v
    return out


@lru_cache(None)
def reduced_pairs_of_degree(d: int):
  '''
  Helper function for generating reduced pairs of degree d
  '''
  pv = partitions_by_largest_vec(d)
  pairs = []
  for i in range(len(pv) - 1, -1, -1):
      for pi in pv[i]:
          ub = d - i
          if ub < 0:
              continue
          for j in range(ub, -1, -1):
              for sig in pv[j]:
                  pairs.append((sig, pi))
  return pairs


@lru_cache(None)
def all_reduced_pairs_up_to_d(d: int) -> list:
  '''
  For a given d, returns all reduced pairs up to that degree
  '''
  seen = set()
  out = []
  for d0 in range(1, d + 1):
      for sig, pi in reduced_pairs_of_degree(d0):
          key = (sig, pi)
          if key not in seen:
              seen.add(key)
              out.append(key)
  return out


# ----------------------------
# Growth potential + A
# ----------------------------

@lru_cache(None)
def growth_potential(sig, pi):
    d = sum(sig)
    g = 0
    for s in sig:
        for p in pi:
            if s + p - d == 0:
                g += 1
    return g


@lru_cache(None)
def A_cached(sig, pi, d):
    dp = sum(sig)
    g = growth_potential(sig, pi)

    if len(sig) == 2 and len(pi) == 2:
        return 4 * (d - dp) + (1 if d == dp else 0)

    if g >= 1:
        return math.comb((d - dp) + g - 1, g - 1)
    else:
        return 1 if d == dp else 0


# ----------------------------
# Multiplicity
# ----------------------------

@lru_cache(None)
def binom(n, k):
    return math.comb(n, k)


@lru_cache(None)
def N_sigma_pi_cached(m, d, sig, pi):
    a = A_cached(sig, pi, d)
    ls = sum(1 for x in sig if x != 0)
    lp = sum(1 for x in pi if x != 0)

    if sig != pi:
        return a * 2 * binom(m, ls) * binom(m, lp)
    else:
        return a * binom(m, ls) ** 2


# ----------------------------
# N0
# ----------------------------

@lru_cache(None)
def N0(m, d):
    return (
        math.comb(d + m - 1, d) * m
        + math.comb(d + m - 1, d) * m
        - m * m
    )


# ----------------------------
# Block decomposition
# ----------------------------

@lru_cache(None)
def padded(sig, pi):
    n = max(len(sig), len(pi))
    return (
        tuple(sig) + (0,) * (n - len(sig)),
        tuple(pi) + (0,) * (n - len(pi)),
    )


@lru_cache(None)
def RSK_block(sig, pi):
    return RSK_pi_sig((list(sig), list(pi)))


def block_decomp(m, n, d):
    trace_total = N0(m, d)
    det_total = 1

    seen = set()

    for sig, pi in all_reduced_pairs_up_to_d(d):
        if len(sig) > m or len(pi) > n:
            continue

        key = tuple(sorted((sig, pi)))
        if key in seen:
            continue
        seen.add(key)

        sig_p, pi_p = padded(sig, pi)
        N = N_sigma_pi_cached(m, d, sig_p, pi_p)

        if N == 0:
            continue

        R = RSK_block(sig_p, pi_p)
        tr = int(np.trace(R))
        det = int(round(np.linalg.det(R)))

        trace_total += N * tr
        det_total *= det ** N

    return trace_total, det_total

tr, det = block_decomp(3, 3, 3)
print(f"m: {m}, d: {d}, Trace: {tr}, Det: {det}")

m: 3, d: 4, Trace: 42, Det: -1


In [ ]:
trace_det_tables(3,3)

([[None, None, None, None],
  [None, 1, 1, 1],
  [None, 4, 8, 12],
  [None, 9, 27, 42]],
 [[None, None, None, None],
  [None, 1, 1, 1],
  [None, 1, -1, 1],
  [None, 1, -1, -1]])

m: 3
n: 5
d: 9
Trace: 660.0
Determinant: 0.9999999999999918


m = 3:
*   d = 8
  *   1m
  *   1
  *   270
*   d = 9
  *   5m
  *   1
  *   660

*   d = 10
  *

*   List item



        d=1       d=2        d=3        d=4         d=5         d=6          d=7
m=1        1         1          1          1           1           1            1
m=2        4         8         12         17          24          32           40
m=3        9        27         42         70         160         241          203
m=4       16        64         48        -33         613        1107        -4186
m=5       25       125       -175      -1650        2853       14259      -68238
m=6       36       216      -1164      -9824       19231      178667     -521810
m=7       49       343      -4018     -38563      114954     1423009


## Problem 1.6 Code

Find a(ll) reduced pairs that have real eigenvalues only

In [ ]:
def search_reduced_pairs(d: int):
  for sig, pi in all_reduced_pairs_up_to_d(d):
    if len(sig) < 3 or len(pi) < 3:
        # already been found
        continue

    sig_p, pi_p = padded(sig, pi)
    R = RSK_block(sig_p, pi_p)
    eigvals = np.linalg.eigvals(R)

    if np.all(np.isreal(eigvals)):
        print("Found candidate:", sig, pi, "Eigenvalues:", eigvals)


In [ ]:
search_reduced_pairs(2)

## Problem 1.7 Code

In [ ]:
# def non_triangular(pi, sig) -> bool:
#   if pi.ndim >
#   if len(sig) == 2 and pi[0] == sig[0]:
#     return False
#   return True

# Proposition 4.1 for RSK 1^d, 1^d


# Orthogonal RSK

Problem 2.3

In [ ]:
from itertools import combinations_with_replacement

def weak_compositions(total, parts):
    """Generate all weak compositions of `total` into `parts` parts."""
    if parts == 1:
        yield (total,)
        return
    for i in range(total + 1):
        for rest in weak_compositions(total - i, parts - 1):
            yield (i,) + rest


def gen_all_M(n: int, d: int):

    if d % 2 != 0:
        return []

    k = d // 2
    results = []

    upper_coords = [(i, j) for i in range(n) for j in range(i+1, n)]
    upper_parts = len(upper_coords)

    # diagonal sum can range from 0 to k
    for diag_sum in range(k + 1):

        # choose diagonal partition of size diag_sum
        for diag_comp in weak_compositions(diag_sum, n):

            remaining = k - diag_sum

            # distribute remainder to upper triangle
            for upper_comp in weak_compositions(remaining, upper_parts):

                sparse = []

                # diagonals
                for i in range(n):
                    val = 2 * diag_comp[i]
                    if val:
                        sparse.append((val, i, i))

                # off-diagonal
                for idx, (i, j) in enumerate(upper_coords):
                    val = upper_comp[idx]
                    if val:
                        sparse.append((val, i, j))
                        sparse.append((val, j, i))

                sparse.sort(key=lambda x: (x[2], -x[1]))
                results.append(sparse)

    return results

In [ ]:
# sanity check test
mats = gen_all_M(2, 8)
for m in mats:
    print(m)

[(4, 1, 0), (4, 0, 1)]
[(3, 1, 0), (2, 1, 1), (3, 0, 1)]
[(3, 1, 0), (2, 0, 0), (3, 0, 1)]
[(2, 1, 0), (4, 1, 1), (2, 0, 1)]
[(2, 1, 0), (2, 0, 0), (2, 1, 1), (2, 0, 1)]
[(2, 1, 0), (4, 0, 0), (2, 0, 1)]
[(1, 1, 0), (6, 1, 1), (1, 0, 1)]
[(1, 1, 0), (2, 0, 0), (4, 1, 1), (1, 0, 1)]
[(1, 1, 0), (4, 0, 0), (2, 1, 1), (1, 0, 1)]
[(1, 1, 0), (6, 0, 0), (1, 0, 1)]
[(8, 1, 1)]
[(2, 0, 0), (6, 1, 1)]
[(4, 0, 0), (4, 1, 1)]
[(6, 0, 0), (2, 1, 1)]
[(8, 0, 0)]


In [ ]:
sparse_M = [
    (2, 0, 0),  # M[0][0] = 2
    (1, 0, 1),  # M[0][1] = 1
    (1, 1, 0),  # M[1][0] = 1
    (2, 1, 1),  # M[1][1] = 2
    (1, 1, 2),  # M[1][2] = 1
    (1, 2, 1),  # M[2][1] = 1
]
# sparse_M = [
#     (1, 1, 0),
#     (1, 0, 1),
#     (2, 1, 1)
# ]

In [ ]:
def determinant_symmetric(T, z_matrix):
    num_cols = max(len(row) for row in T)
    cols = []
    for c in range(num_cols):
        col = [T[r][c] for r in range(len(T)) if c < len(T[r])]
        cols.append(col)

    product = 1
    for k in range(0, len(cols), 2):
        row_indices = cols[k]
        col_indices = cols[k + 1]
        size = len(row_indices)
        minor = Matrix(size, size, lambda a, b:
            z_matrix[min(row_indices[a], col_indices[b]) - 1,
                     max(row_indices[a], col_indices[b]) - 1]
        )
        product *= minor.det()

    return expand(product)

In [ ]:
def generate_monomials_symmetric(sparse_list, n):
    z = Matrix(n, n, lambda i, j: symbols(f"z{min(i,j)+1}{max(i,j)+1}"))

    monomials = []
    idx_dict = {}
    exp_dict = {}

    for idx, sparse in enumerate(sparse_list):
        mon = 1
        temp_idx = []
        table_dict = {}

        for val, i, j in sparse:
          ci, cj = min(i, j), max(i, j)
          var = z[ci, cj]
          if i == j:
              exp = val // 2  # diagonal entries are stored as 2x, reduce back
          elif i < j:
              exp = val       # upper triangle only, skip lower
          else:
              exp = 0
          if exp > 0:
              pair = (ci + 1, cj + 1)
              temp_idx.extend([pair] * exp)
              mon *= var**exp
              table_dict[var] = table_dict.get(var, 0) + exp

        exp_dict[idx] = table_dict
        idx_dict[idx] = temp_idx
        monomials.append(mon)

    return monomials, idx_dict, exp_dict, z

In [ ]:
def col_insert_get_row(T_ins, x):
    bumped = x
    for c in range(len(T_ins)):
        col = T_ins[c]
        idx = next((i for i, v in enumerate(col) if v >= bumped), None)  # >= for dual insertion
        if idx is None:
            col.append(bumped)
            return T_ins, len(col) - 1
        else:
            bumped, T_ins[c][idx] = T_ins[c][idx], bumped
    T_ins.append([bumped])
    return T_ins, 0

def dual_rsk(sparse_M):
    if not sparse_M:
        return []

    sorted_M = sorted(sparse_M, key=lambda x: (x[2], -x[1]))  # col asc, row desc

    biword = []
    for val, i, j in sorted_M:
        biword.extend([(i + 1, j + 1)] * val)  # full val, no halving, no i<=j filter

    T_ins = []
    for (x, _) in biword:
        T_ins, _ = col_insert_get_row(T_ins, x)

    rows = {}
    for col in T_ins:
        for r, val in enumerate(col):
            rows.setdefault(r, []).append(val)

    tableau = []
    for r in sorted(rows.keys()):
        tableau.append(rows[r])

    return tableau

In [ ]:
from scipy.sparse import csc_matrix

def generate_final_sparse_matrix(monomial_mapping, determinant_list):
    rows, cols, data = [], [], []
    size = len(monomial_mapping)

    for col_idx, curr_list in enumerate(determinant_list):
        for term in curr_list.as_ordered_terms():
            coeff, monomial = term.as_coeff_Mul()

            if monomial in monomial_mapping:
                row_idx = monomial_mapping[monomial]
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(float(coeff))
            elif -monomial in monomial_mapping:
                row_idx = monomial_mapping[-monomial]
                rows.append(row_idx)
                cols.append(col_idx)
                data.append(float(-coeff))

    return csc_matrix((data, (rows, cols)), shape=(size, size))

In [ ]:
def orth_rsk(n, d):
  sparse_list = gen_all_M(n, d)[::-1]
  monomials, idx_dict, exp_dict, z = generate_monomials_symmetric(sparse_list, n)
  monomial_mapping = {mon: idx for idx, mon in enumerate(monomials)}
  tableaux = [dual_rsk(m) for m in sparse_list]
  determinant_list = [determinant_symmetric(P, z) for P in tableaux]

  return generate_final_sparse_matrix(monomial_mapping, determinant_list)

def test_printing(sparse_list, tableaux, determinant_list):
  for i, (s, t, det) in enumerate(zip(sparse_list, tableaux, determinant_list)):
    print(f"Matrix {i + 1}")
    print(f"Sparse Matrix: {s}")
    print(f"Tableaux: \n{t}")
    print(f"Polynomial \n{det}\n")

In [ ]:
import sys
import numpy
numpy.set_printoptions(threshold=sys.maxsize)
m = orth_rsk(2, 8)
print(m)

<Compressed Sparse Column sparse matrix of dtype 'float64'
	with 22 stored elements and shape (15, 15)>
  Coords	Values
  (0, 0)	1.0
  (1, 1)	1.0
  (9, 1)	-1.0
  (2, 2)	1.0
  (10, 2)	-2.0
  (14, 2)	1.0
  (3, 3)	1.0
  (11, 3)	-1.0
  (4, 4)	1.0
  (5, 5)	1.0
  (6, 6)	1.0
  (12, 6)	-1.0
  (7, 7)	1.0
  (13, 7)	-1.0
  (8, 8)	1.0
  (1, 9)	1.0
  (2, 10)	1.0
  (10, 10)	-1.0
  (3, 11)	1.0
  (6, 12)	1.0
  (7, 13)	1.0
  (2, 14)	1.0


In [ ]:
# testing suite for orth matrix
from scipy.sparse.linalg import splu

# for n in range(1, 4):
#     for d in range(2, 8, 2):
#         sparse_m = orth_rsk(n, d)
#         trace = sparse_m.diagonal().sum()
#         lu = splu(sparse_m.tocsc())
#         det = np.prod(lu.U.diagonal())
#         print(f"n={n}, d={d}: \ndet={det:.4f}\n trace={trace:.4f}")

In [ ]:
n = 3
for d in range(2, 12, 2):
  sparse_m = orth_rsk(n, d)
  trace = sparse_m.diagonal().sum()
  lu = splu(sparse_m.tocsc())
  det = np.prod(lu.U.diagonal())
  print(f"n={n}, d={d}: \ndet={det:.4f}\ntrace={trace:.4f}")

n=3, d=2: 
det=1.0000
trace=6.0000
n=3, d=4: 
det=-1.0000
trace=15.0000
n=3, d=6: 
det=1.0000
trace=30.0000
n=3, d=8: 
det=-1.0000
trace=42.0000
n=3, d=10: 
det=-1.0000
trace=51.0000


In [ ]:
def row_weight(sparse_M, n):
    sums = [0] * n
    for val, i, j in sparse_M:
        sums[i] += val
    return tuple(sums)

def gen_all_M_sorted(n, d):
    """gen_all_M but sorted by row weight."""
    matrices = gen_all_M(n, d)
    matrices.sort(key=lambda m: row_weight(m, n))
    return matrices

def orth_rsk_sigma(n, sigma):
    """Run orth_rsk on just the matrices with row weight sigma."""
    d = sum(sigma)
    matrices = [m for m in gen_all_M(n, d) if row_weight(m, n) == sigma]
    return orth_rsk_from_matrices(n, matrices)

In [ ]:
def orth_rsk(n, d):
    sparse_list = sorted(gen_all_M(n, d), key=lambda m: row_weight(m, n))
    monomials, idx_dict, exp_dict, z = generate_monomials_symmetric(sparse_list, n)
    monomial_mapping = {mon: idx for idx, mon in enumerate(monomials)}
    tableaux = [dual_rsk(m) for m in sparse_list]
    determinant_list = [determinant_symmetric(P, z) for P in tableaux]
    return generate_final_sparse_matrix(monomial_mapping, determinant_list)

def orth_rsk_sigma(n, sigma):
    d = sum(sigma)
    sparse_list = [m for m in gen_all_M(n, d) if row_weight(m, n) == sigma]
    monomials, idx_dict, exp_dict, z = generate_monomials_symmetric(sparse_list, n)
    monomial_mapping = {mon: idx for idx, mon in enumerate(monomials)}
    tableaux = [dual_rsk(m) for m in sparse_list]
    determinant_list = [determinant_symmetric(P, z) for P in tableaux]
    return generate_final_sparse_matrix(monomial_mapping, determinant_list)

# Orthogonal RSK 2nd Algorithm

In [ ]:
def gen_all_Mp(n: int, d: int):

    if d % 2 != 0:
        return []

    k = d // 2
    results = []

    upper_coords = [(i, j) for i in range(n) for j in range(i+1, n)]
    upper_parts = len(upper_coords)

    # diagonal sum can range from 0 to k
    for diag_sum in range(k + 1):

        # choose diagonal partition of size diag_sum
        for diag_comp in weak_compositions(diag_sum, n):

            remaining = k - diag_sum

            # distribute remainder to upper triangle
            for upper_comp in weak_compositions(remaining, upper_parts):

                sparse = []

                # diagonals
                for i in range(n):
                    val = 2 * diag_comp[i]
                    if val:
                        sparse.append((val, i, i))

                # off-diagonal
                for idx, (i, j) in enumerate(upper_coords):
                    val = upper_comp[idx]
                    if val:
                        sparse.append((val, i, j))

                sparse.sort(key=lambda x: (x[2], -x[1]))
                results.append(sparse) # assume the ordering is correct(?)
    return results

In [ ]:
import numpy as np

def col_insert_get_row(tableau, x):
    """
    Column-inserts x into tableau T (column by column).
    Returns the updated tableau and the row index where bumping stopped.

    Rule: in each column, bump the smallest y >= x if it exists;
    otherwise append x to the bottom of the column and stop.
    """
    tableau = tableau.copy()
    rows, cols = tableau.shape
    bumped = x

    for c in range(cols):
        col = tableau[:, c]
        occupied = np.sum(col > 0)
        eligible_vals = col[col > 0]

        # Find smallest entry >= bumped in this column
        idx = np.where(eligible_vals >= bumped)[0]

        if idx.size == 0:
            # No entry >= bumped — append bumped to the bottom of this column and stop
            if occupied < rows:
                tableau[occupied, c] = bumped
                return tableau, int(occupied)
            else:
                # Column full — expand (rare, handled below)
                continue
        else:
            # Bump the smallest eligible entry
            i = idx[0]
            bumped, tableau[i, c] = tableau[i, c], bumped

    # Fell off the right — append a new column
    new_col = np.zeros((tableau.shape[0], 1), dtype=int)
    new_col[0, 0] = bumped
    tableau = np.hstack([tableau, new_col])
    return tableau, 0


def ortho_rsk_second(M_upper):
    """
    Orthogonal RSK (second/streamlined algorithm).

    Given an upper-triangular nonneg integer matrix M', builds a biword
    by reading up each column from left to right (bottom-left to top-right),
    then processes each biletter (x, x') as follows:
      1. Column-insert x into T -> determines stopping row i_x
      2. Append x' to the END of row i_x in T

    Returns the final tableau T.
    """
    n = M_upper.shape[0]

    # --- Build biword: read up columns left to right ---
    biword = []
    for col in range(n):
        for row in range(n - 1, -1, -1):  # bottom to top
            for _ in range(M_upper[row, col]):
                biword.append((row + 1, col + 1))  # 1-indexed

    # --- Process each biletter ---
    T = np.zeros((n, n), dtype=int)

    for (x, x_prime) in biword:
        # Step 1: column-insert x, get the stopping row
        T, stop_row = col_insert_get_row(T, x)

        # Step 2: append x' at the end of row stop_row
        row_data = T[stop_row]
        append_pos = np.sum(row_data > 0)

        # Expand T if needed
        if append_pos >= T.shape[1]:
            T = np.hstack([T, np.zeros((T.shape[0], 1), dtype=int)])

        T[stop_row, append_pos] = x_prime

    return T


def print_tableau(tableau, name='T'):
    """For debugging — prints only non-zero rows."""
    print(f"{name}:")
    for row in tableau:
        row_nonzero = row[row > 0]
        if row_nonzero.size > 0:
            print(row_nonzero)

In [ ]:
results = gen_all_Mp(2, 4)
for m in results:
    print(m)

[(2, 0, 1)]
[(2, 1, 1), (1, 0, 1)]
[(2, 0, 0), (1, 0, 1)]
[(4, 1, 1)]
[(2, 0, 0), (2, 1, 1)]
[(4, 0, 0)]
